In [ ]:
import pyreadr
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor


#carga de datos
result = pyreadr.read_r('data/listings.RData')
df = result['listings']

print("Shape inicial:", df.shape)
df.head()

Shape inicial: (171748, 80)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,city
0,5456.0,https://www.airbnb.com/rooms/5456,2.025092e+13,2025-09-17,city scrape,"Walk to 6th, Rainey St and Convention Ctr",Great central location for walking to Convent...,My neighborhood is ideally located if you want...,https://a0.muscache.com/pictures/14084884/b5a3...,8028,...,4.73,4.79,NaN,f,1,1,0,0,3.52,"Austin, Texas"
1,6448.0,https://www.airbnb.com/rooms/6448,2.025092e+13,2025-09-17,city scrape,"Secluded Studio @ Zilker - King Bed, Bright & ...","Clean, private space with everything you need ...",The neighborhood is fun and funky (but quiet)!...,https://a0.muscache.com/pictures/airflow/Hosti...,14156,...,4.97,4.88,NaN,t,1,1,0,0,1.98,"Austin, Texas"
2,8502.0,https://www.airbnb.com/rooms/8502,2.025092e+13,2025-09-17,city scrape,Woodland Studio Lodging,Studio rental on lower level of home located i...,,https://a0.muscache.com/pictures/miso/Hosting-...,25298,...,4.69,4.63,NaN,f,1,1,0,0,0.28,"Austin, Texas"
3,13035.0,https://www.airbnb.com/rooms/13035,2.025092e+13,2025-09-17,city scrape,Historic house in highly walkable East Austin,Comfortable 2 bedroom/2 bathroom home very cen...,East Cesar Chavez is a gentrifying urban area ...,https://a0.muscache.com/pictures/miso/Hosting-...,50793,...,5.00,4.95,NaN,f,2,2,0,0,0.11,"Austin, Texas"
4,22828.0,https://www.airbnb.com/rooms/22828,2.025092e+13,2025-09-16,city scrape,Garage Apartment central SE Austin,"Fully furnished, centrally located, second sto...","wikipedia: East_Riverside-Oltorf,_Austin,_Texas",https://a0.muscache.com/pictures/miso/Hosting-...,56488,...,4.72,4.84,NaN,f,1,1,0,0,0.30,"Austin, Texas"


In [15]:
# Limpiar price (SIN regex complicado)
df["price"] = df["price"].astype(str).str.replace("$", "", regex=False)
df["price"] = df["price"].str.replace(",", "", regex=False)
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# Ver antes de borrar
print("Antes de dropna:", df.shape)
print("NaN en price:", df["price"].isna().sum())

Antes de dropna: (171748, 80)
NaN en price: 95502


In [16]:
print("Después de limpieza:", df.shape)
print(df["price"].describe())

Después de limpieza: (171748, 80)
count    76246.000000
mean       750.509220
std       4250.606945
min          8.000000
25%        120.000000
50%        193.000000
75%        326.000000
max      50123.000000
Name: price, dtype: float64


In [33]:
# 🔹 Variables numéricas
df_numeric = df.select_dtypes(include=['int64', 'float64'])

# 🔹 Eliminar NaN en TODO el dataset numérico (esto sí es correcto aquí)
df_model = df_numeric.dropna()

print("Shape limpio:", df_model.shape)
print("NaN totales:", df_model.isna().sum().sum())

Shape limpio: (110096, 15)
NaN totales: 0


In [35]:
print("price en df:", "price" in df.columns)
print("price en df_numeric:", "price" in df_numeric.columns)

price en df: True
price en df_numeric: False


In [34]:
y = df_model["price"]
X = df_model.drop(columns=["price"])

KeyError: 'price'

In [20]:
y = df_numeric["price"]
X = df_numeric.drop(columns=["price"])

In [29]:
# Unir X e y para limpiar juntos
df_model = df_numeric.dropna()

print("Shape después de eliminar NaN:", df_model.shape)

Shape después de eliminar NaN: (62820, 16)


In [30]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

c:\Users\Elvir\OneDrive\Desktop\env\Lib\site-packages\numpy\_core\fromnumeric.py:83: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
c:\Users\Elvir\OneDrive\Desktop\env\Lib\site-packages\sklearn\utils\extmath.py:1213: RuntimeWarning: invalid value encountered in subtract
  temp = X - T
c:\Users\Elvir\OneDrive\Desktop\env\Lib\site-packages\numpy\_core\fromnumeric.py:83: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
c:\Users\Elvir\OneDrive\Desktop\env\Lib\site-packages\sklearn\preprocessing\_data.py:1115: RuntimeWarning: invalid value encountered in subtract
  X -= xp.astype(self.mean_, X.dtype)


In [31]:

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [32]:

knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train, y_train)

ValueError: Input X contains NaN.
KNeighborsRegressor does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values